# Woche 4: Kurzzeit-Fourier-Transformation

Die Kurzzeit-Fourier-Transformation (STFT) erweitert die DFT auf Signale, deren Frequenzinhalt sich ueber die Zeit aendert. Das Ergebnis ist ein Spektrogramm.

## Lernziele

Nach dieser Woche sollten Sie:

- erklaeren koennen, warum eine normale DFT fuer zeitveraenderliche Signale nicht ausreicht
- den Ablauf der STFT beschreiben koennen
- Fensterlaenge, Hopgroesse und Ueberlappung erklaeren koennen
- ein Spektrogramm interpretieren koennen
- den Zielkonflikt zwischen Zeit- und Frequenzaufloesung kennen
- grob verstehen, wie eine ISTFT ein Zeitsignal rekonstruiert

## Problem der normalen DFT

Eine normale DFT betrachtet ein komplettes Signal oder einen grossen Block auf einmal. Sie sagt, welche Frequenzen insgesamt vorkommen.

Wenn sich die Frequenz aber im Verlauf des Signals aendert, fehlt die Zeitinformation. Ein Sweep ist ein gutes Beispiel: Die Frequenz steigt mit der Zeit. Die DFT zeigt dann viele Frequenzen, aber nicht, wann sie aufgetreten sind.

## Idee der STFT

Die STFT zerlegt das Signal in kurze Zeitfenster. Fuer jedes Fenster wird eine DFT berechnet.

Ablauf:

1. Fensterfunktion $w(n)$ waehlen.
2. Ersten Signalblock ausschneiden.
3. Block mit Fenster multiplizieren.
4. DFT berechnen.
5. Fenster um die Hopgroesse verschieben.
6. Wiederholen, bis das Signal verarbeitet ist.

Die DFT-Ergebnisse werden als Spalten in einer Matrix gespeichert. Diese Matrix ist das Spektrogramm.

## Fensterfunktion

Eine Fensterfunktion blendet einen Block weich ein und aus. Ohne Fenster wuerde der Block abrupt abgeschnitten. Das kann kuenstliche Frequenzanteile erzeugen.

Ein haeufiges Fenster ist das Hann-Fenster:

$w(n)=0.5\cdot\left(1-\cos\left(2\pi\frac{n+0.5}{N}\right)\right)$

Das Fenster reduziert harte Spruenge an den Blockgrenzen.

In [ ]:
import numpy as np

def hann_window(N):
    return 0.5 * (1 - np.cos(2 * np.pi * (np.arange(N) + 0.5) / N))

w = hann_window(16)
w

## Fensterlaenge und Hopgroesse

Die Fensterlaenge bestimmt, wie viele Samples pro Analyseblock betrachtet werden.

Die Hopgroesse bestimmt, wie viele Samples das Fenster pro Schritt weitergeschoben wird.

Beispiel:

- Fensterlaenge: 1024 Samples
- Hopgroesse: 512 Samples
- Ueberlappung: 50 Prozent

Eine kleinere Hopgroesse bedeutet mehr Spalten im Spektrogramm und bessere zeitliche Abtastung, aber auch mehr Rechenaufwand.

In [ ]:
signal_length = 48000
window_size = 1024
hop_size = 512
number_of_columns = int((signal_length - window_size) / hop_size) + 1
number_of_columns

## Zeit-Frequenz-Zielkonflikt

Es gibt einen grundlegenden Zielkonflikt:

- kurzes Fenster: gute Zeitaufloesung, schlechte Frequenzaufloesung
- langes Fenster: gute Frequenzaufloesung, schlechte Zeitaufloesung

Das liegt daran, dass eine genaue Frequenzanalyse genug Zeit braucht. Sehr kurze Signalausschnitte enthalten weniger Information ueber tiefe oder nahe beieinanderliegende Frequenzen.

## Spektrogramm interpretieren

Ein Spektrogramm hat typischerweise:

- x-Achse: Zeit bzw. Blocknummer
- y-Achse: Frequenz
- Farbe/Helligkeit: Betrag oder Leistung der Frequenz

Helle Bereiche bedeuten starke Frequenzanteile. Bei Sprache sieht man oft zeitlich veraenderliche Baender und Energie in bestimmten Frequenzbereichen.

In [ ]:
# Kleine STFT-Demo ohne Plot: Ergebnisform berechnen.
r = 8000
N = r
x = np.sin(2 * np.pi * 440 * np.arange(N) / r)
window_size = 256
hop_size = 128
K = 256
w = hann_window(window_size)

columns = []
for start in range(0, len(x) - window_size + 1, hop_size):
    block = x[start:start + window_size] * w
    spectrum = np.fft.rfft(block, n=K)
    columns.append(np.abs(spectrum))

spectrogram = np.array(columns).T
spectrogram.shape

## ISTFT grob erklaert

Die inverse STFT rekonstruiert aus den Spektren wieder ein Zeitsignal.

Prinzip:

1. Fuer jede Spektrogrammspalte inverse DFT berechnen.
2. Zeitblock mit Synthesefenster multiplizieren.
3. Bloecke an der passenden Position addieren.

Dieses Addieren ueberlappender Bloecke heisst Overlap-Add.

## Typische Fehler

- Fensterlaenge und Hopgroesse verwechseln.
- Spektrogrammachsen falsch interpretieren.
- Zu kleines Fenster waehlen und schlechte Frequenzaufloesung erhalten.
- Zu grosses Fenster waehlen und schnelle Aenderungen verlieren.
- Vergessen, dass STFT-Werte komplex sind und oft der Betrag dargestellt wird.

## Mini-Check

1. Warum reicht eine normale DFT fuer einen Sweep nicht aus?
2. Was ist ein Spektrogramm?
3. Was macht ein Hann-Fenster?
4. Was passiert, wenn die Fensterlaenge groesser wird?
5. Was bedeutet Hopgroesse?